# H06 — 让工程结构反映职责

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H06 — 让工程结构反映职责。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H06-components.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

按钮、采样和通信挤在入口里，改一个外设就影响整个工具。现在才拆模块，避免第一天就面对过度设计的目录。

前置：H05 持续工程和可恢复版本。

## 本次够用的知识

ESP-IDF Component 是构建单元，不必等于一个类。REQUIRES 公开传递依赖，PRIV_REQUIRES 用于仅实现需要的依赖。HAL 表达业务需要的能力，Driver 实现具体外设操作；接口不要泄漏所有芯片 API。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 先画依赖再移动文件 · Guided

保存 H05 工作版本，列出 Controller、Board、Sensor、Protocol、配置与入口的职责。画箭头：业务 → 接口，硬件实现 → 接口；业务不应包含 gpio.h。
选择一个无硬件依赖的策略函数先拆出，别同时重构全部外设。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 创建一个小 Component · Guided

将 status_policy.hpp 放 components/status_policy/include，status_policy.cpp 放 components/status_policy。新 CMakeLists.txt 使用 idf_component_register(SRCS "status_policy.cpp" INCLUDE_DIRS "include")。
main 增加 REQUIRES status_policy。示例验证 connected && !fault；入口只组装和调用。随后把自己已有的 Controller 移到类似组件。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H06/examples/status_policy.hpp`。目标：`components/status_policy/include/status_policy.hpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。

In [ ]:
%%esp32_example --target components/status_policy/include/status_policy.hpp
#pragma once
namespace status_policy {
    bool should_show_ready(bool connected, bool fault);
}


示例来源：`step2/lessons/H06/examples/status_policy.cpp`。目标：`components/status_policy/status_policy.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES status_policy freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target components/status_policy/status_policy.cpp
#include "status_policy.hpp"
namespace status_policy {
    bool should_show_ready(bool connected, bool fault) {
        return connected && !fault;
    }
}


示例来源：`step2/lessons/H06/examples/main.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES status_policy freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "status_policy.hpp"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    ESP_LOGI("policy", "ready=%d", status_policy::should_show_ready(true, false));
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 配置不是散落的常量 · Modify

新增 main/Kconfig.projbuild，写一个 Academy 菜单里的 LED_GPIO int 配置，记录 range 与经典板限制；使用 sdkconfig.h 中生成的 CONFIG_ 名。
menuconfig 选择值后 build，观察 sdkconfig 和公开 defaults 的区别。配置范围合法不等于该脚可用，仍要核对板型。不要把密码作为公共 defaults。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 三种错误分层定位 · Debug

先漏登记 status_policy.cpp 制造 linker error；恢复后从公共头 include 某 Driver 类型但仅声明 PRIV_REQUIRES，观察依赖公开性。
运行错误用日志模块 tag、上下文、esp_err_to_name 定位，不以盲目 fullclean 解决一切。支持 JTAG 的板型可选断点，但别假定每台 DevKit 都有板载 JTAG。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 复用上层测试 · Build

让 Controller 组件不直接依赖 FreeRTOS/GPIO。写一个 host CMake 目标，只编译 Controller 与 fake HAL；保留实机入口。
测试包括输入极性映射之外的业务边界、sensor error、计时取消。故意改一个状态条件，应有至少一项测试失败；仅编译通过不算行为测试。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 整理持续工程 · Build

按实际需要拆 Controller、HAL/Driver、协议适配；说明公开 include 与 private source 的边界。入口负责初始化、所有权和任务生命周期。
验收：新增一个独立组件，登记依赖，保持 host tests、firmware build 与已有硬件行为。记录第三方版本与 dependencies.lock，不复制 managed_components 当源码。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 给未来的自己交接 · Open

请另一个人或新聊天只看 README 和目录树解释模块关系。对照解释是否正确，补缺失的契约。
H07 网络也只是新的适配层；网络回调不应绕过 Controller 直接修改所有硬件。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 新增 Component 与依赖登记可构建
- [ ] host 测试实际运行且能抓住一次人为回归
- [ ] GPIO/协议/业务的所有权与依赖可解释

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H06',
    [
        '新增 Component 与依赖登记可构建',
        'host 测试实际运行且能抓住一次人为回归',
        'GPIO/协议/业务的所有权与依赖可解释',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H06/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-guides/build-system.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-guides/build-system.html)